This notebook highlights using odmlib v0.2 to explore an ODM v2.0 CRF generated as part of 360i. It walks through the hierarchical structure in a basic way using some simple odmlib features. The ODM v2.0 CRF uses some of the new v2.0 features as well as representing new content like Biomedical Concepts.

Import odmlib to begin processing the ODM v2.0 CRF file.

In [1]:
from odmlib import odm_loader as OL, loader as LO

Next use odmlib to load the ODM v2.0 XML file and create variables for the document root and MetaDataVersion elements.

In [2]:
odm_file = "data/vital_signs_odmv2-0.xml"
loader = LO.ODMLoader(OL.XMLODMLoader(model_package='odm_2_0', ns_uri='http://www.cdisc.org/ns/odm/v2.0'))
loader.open_odm_document(odm_file)
odm = loader.root()
mdv = loader.MetaDataVersion()

Having loaded the ODM file, print out the names of the Study and the MetaDataVersion.

In [3]:
print(f"Study Name: {odm.Study[0].StudyName}")
print(f"MetaDataVersionName: {mdv.Name}")

Study Name: Vital Signs
MetaDataVersionName: Vital Signs


Now, let's start exploring the ItemGroupDefs to see the different types that are included.

In [4]:
print("Listing the Item Groups in the ODM v2.0 file:")
for igd in mdv.ItemGroupDef:
    print(f"    Name: {igd.Name} and Type: {igd.Type}")

Listing the Item Groups in the ODM v2.0 file:
    Name: Vital Signs and Type: Form
    Name: Vital Signs Performed and Type: Section
    Name: Vital Signs and Type: Section
    Name: Vital Signs Performed and Type: Concept
    Name: Systolic Blood Pressure (Denormalized) and Type: Concept
    Name: Diastolic Blood Pressure (Denormalized) and Type: Concept
    Name: Height (Denormalized) and Type: Concept
    Name: Weight (Denormalized) and Type: Concept
    Name: Body Mass Index (Denormalized) and Type: Concept
    Name: Pulse (Denormalized) and Type: Concept
    Name: Respiratory Rate (Denormalized) and Type: Concept
    Name: Temperature (Denormalized) and Type: Concept
    Name: Heart Rate (Denormalized) and Type: Concept


The above listing shows that we have ItemGroupDef types of Form, Section, and Concept. The content of the ItemGroupDefs indicate that this is a hierarchical structure: Form -> Sections -> Concepts. The Concept ItemGroupDef includes the references to the ItemRefs, or variables. The remaining code blocks, prior to the validation checks, navigate this hierarchy while printing content.

The next code block prints out the name of the Form.

In [5]:
igd_forms = mdv.find_all("ItemGroupDef", "Type", "Form")
for form in igd_forms:
    print(f"Name: {form.Name}")

Name: Vital Signs


The next two blocks list the Sections found within the Form. The following block lists the Section OIDs.

In [6]:
# create a list of the sections within the form
section_refs = igd_forms[0].ItemGroupRef
# list the OIDs for the ItemGroupRefs
for section_ref in section_refs:
    print(f"OID: {section_ref.ItemGroupOID}")

OID: IG.VS_01_1
OID: IG.VS_02_2


This block lists the Section name and ItemGroupDef Type.

In [7]:
# get the form section ItemGroupDefs
igd_sections = []
for section_ref in section_refs:
    igd_sections.append(mdv.find("ItemGroupDef", "OID", section_ref.ItemGroupOID))

# list the names of the ItemGroupDef sections
for igd in igd_sections:
    print(f"Name: {igd.Name} and Type: {igd.Type}")

Name: Vital Signs Performed and Type: Section
Name: Vital Signs and Type: Section


Having saved the sections, we now identify and list the Concept ItemGroupDefs within the Sections.

In [8]:
# create a list of all the ItemGroupRefs within the Sections
igd_refs = []
for igd_section in igd_sections:
    igd_refs.extend(igd_section.ItemGroupRef)

# print the OID for each Concept ItemGroupDef
print("Section ItemGroupDef OIDs:")
igd_oids = []
for igd_ref in igd_refs:
    print(f"    {igd_ref.ItemGroupOID}")
    igd_oids.append(igd_ref.ItemGroupOID)

Section ItemGroupDef OIDs:
    IG.VS_01_1_VSPERF_1
    IG.VS_02_2_SYSBP_DENORMALIZED_1
    IG.VS_02_2_DIABP_DENORMALIZED_2
    IG.VS_02_2_HEIGHT_DENORMALIZED_3
    IG.VS_02_2_WEIGHT_DENORMALIZED_4
    IG.VS_02_2_BMI_DENORMALIZED_5
    IG.VS_02_2_PULSE_DENORMALIZED_6
    IG.VS_02_2_RESP_DENORMALIZED_7
    IG.VS_02_2_TEMP_DENORMALIZED_8
    IG.VS_02_2_HR_DENORMALIZED_9


In this next block we look up the ItemGroupDef Concepts referenced in the Sections and print out their names.

In [9]:
# look-up the ItemGroupDef concepts that make up each ItemGroupDef section
igd_concepts = []
print("Concept ItemGroupDef Names:")
for igd_oid in igd_oids:
    igd_concept = mdv.find("ItemGroupDef", "OID", igd_oid)
    print(f"    {igd_concept.Name}")
    igd_concepts.append(igd_concept)

Concept ItemGroupDef Names:
    Vital Signs Performed
    Systolic Blood Pressure (Denormalized)
    Diastolic Blood Pressure (Denormalized)
    Height (Denormalized)
    Weight (Denormalized)
    Body Mass Index (Denormalized)
    Pulse (Denormalized)
    Respiratory Rate (Denormalized)
    Temperature (Denormalized)
    Heart Rate (Denormalized)


Using the list of ItemGroupDef Concepts we next build a dictionary of the item OIDs for each Concept.

In [10]:
concept_variables = {}
for concept in igd_concepts:
    print(f"Concept Name: {concept.Name}")
    concept_variables[concept.Name] = []
    for item_ref in concept:
        print(f"    ItemRef OID: {item_ref.ItemOID}")
        concept_variables[concept.Name].append(item_ref.ItemOID)

Concept Name: Vital Signs Performed
    ItemRef OID: IT.VS_01_1_VSPERF_1.VSPERF
    ItemRef OID: IT.VS_01_1_VSPERF_1.VSDAT
Concept Name: Systolic Blood Pressure (Denormalized)
    ItemRef OID: IT.VS_02_2_SYSBP_DENORMALIZED_1.VSDAT
    ItemRef OID: IT.VS_02_2_SYSBP_DENORMALIZED_1.SYSBP_VSPOS
    ItemRef OID: IT.VS_02_2_SYSBP_DENORMALIZED_1.SYSBP_VSLOC
    ItemRef OID: IT.VS_02_2_SYSBP_DENORMALIZED_1.SYSBP_VSORRES
    ItemRef OID: IT.VS_02_2_SYSBP_DENORMALIZED_1.SYSBP_VSORRESU
Concept Name: Diastolic Blood Pressure (Denormalized)
    ItemRef OID: IT.VS_02_2_DIABP_DENORMALIZED_2.VSDAT
    ItemRef OID: IT.VS_02_2_DIABP_DENORMALIZED_2.DIABP_VSPOS
    ItemRef OID: IT.VS_02_2_DIABP_DENORMALIZED_2.DIABP_VSLOC
    ItemRef OID: IT.VS_02_2_DIABP_DENORMALIZED_2.DIABP_VSORRES
    ItemRef OID: IT.VS_02_2_DIABP_DENORMALIZED_2.DIABP_VSORRESU
Concept Name: Height (Denormalized)
    ItemRef OID: IT.VS_02_2_HEIGHT_DENORMALIZED_3.VSDAT
    ItemRef OID: IT.VS_02_2_HEIGHT_DENORMALIZED_3.HEIGHT_VSORRES
    I

Using the item OIDs, we look up the definition of each item, an ItemDef, and list some details about the variable definition.

In [11]:
for concept, variables in concept_variables.items():
    print(f"Concept Name: {concept}")
    for itd_oid in variables:
        itd = mdv.find("ItemDef", "OID", itd_oid)
        print(f"    Variable Name: {itd.Name}, DataType: {itd.DataType}, Length: {itd.Length}")


Concept Name: Vital Signs Performed
    Variable Name: VSPERF, DataType: text, Length: 1
    Variable Name: VSDAT, DataType: date, Length: 10
Concept Name: Systolic Blood Pressure (Denormalized)
    Variable Name: VSDAT, DataType: date, Length: 10
    Variable Name: SYSBP_VSPOS, DataType: text, Length: 50
    Variable Name: SYSBP_VSLOC, DataType: text, Length: 10
    Variable Name: SYSBP_VSORRES, DataType: integer, Length: 3
    Variable Name: SYSBP_VSORRESU, DataType: text, Length: 10
Concept Name: Diastolic Blood Pressure (Denormalized)
    Variable Name: VSDAT, DataType: date, Length: 10
    Variable Name: DIABP_VSPOS, DataType: text, Length: 50
    Variable Name: DIABP_VSLOC, DataType: text, Length: 50
    Variable Name: DIABP_VSORRES, DataType: integer, Length: 3
    Variable Name: DIABP_VSORRESU, DataType: text, Length: 10
Concept Name: Height (Denormalized)
    Variable Name: VSDAT, DataType: date, Length: 10
    Variable Name: HEIGHT_VSORRES, DataType: decimal, Length: 4
    Va

This next block also processes the ItemDef variable definitions, but this time shows the Question or Prompt associated with the CRF variable.

In [12]:
for concept, variables in concept_variables.items():
    print(f"Concept Name: {concept}")
    for itd_oid in variables:
        itd = mdv.find("ItemDef", "OID", itd_oid)
        if len(itd.Question.TranslatedText):
            print(f"    Question: {" ".join(itd.Question.TranslatedText[0]._content.split())}")
        elif len(itd.Prompt.TranslatedText):
            print(f"    Prompt: {itd.Prompt.TranslatedText[0]._content}")
        else:
            print(f"    No Question or Prompt: {itd.Name}")

Concept Name: Vital Signs Performed
    Question: Were vital signs performed?
    Question: Date of Assessment
Concept Name: Systolic Blood Pressure (Denormalized)
    Question: What was the date of the measurement?
    Question: What was the position of the subject during the Systolic Blood Pressure measurement?
    Question: What was the location of the subject during the Systolic Blood Pressure measurement?
    Question: What was the result of the Systolic Blood Pressure measurement?
    Question: What was the unit of the Systolic Blood Pressure measurement?
Concept Name: Diastolic Blood Pressure (Denormalized)
    Question: What was the date of the measurement?
    Question: What was the position of the subject during the Diastolic Blood Pressure measurement?
    Question: What was the position of the subject during the Diastolic Blood Pressure measurement?
    Question: What was the result of the Diastolic Blood Pressure measurement?
    Question: What was the result of the Diasto

This time we process the ItemDefs again, but highlight the CDASH variable name and SDTM mapping information stored in the Alias element.

In [13]:
for concept, variables in concept_variables.items():
    print(f"Concept Name: {concept}")
    for itd_oid in variables:
        itd = mdv.find("ItemDef", "OID", itd_oid)
        print(f"    Variable Name: {itd.Name}")
        for alias in itd.Alias:
            print(f"        {alias.Context}: {alias.Name}")

Concept Name: Vital Signs Performed
    Variable Name: VSPERF
        SDTM: [NOT SUBMITTED]; VSSTAT = NOT DONE when VSTESTCD = VSALL
        CDASH: VSPERF
    Variable Name: VSDAT
        SDTM: VSDTC
        CDASH: VSDAT
Concept Name: Systolic Blood Pressure (Denormalized)
    Variable Name: VSDAT
        SDTM: VSDTC
        CDASH: VSDAT
    Variable Name: SYSBP_VSPOS
        SDTM: VSPOS when VSTESTCD = SYSBP
        CDASH: SYSBP_VSPOS
    Variable Name: SYSBP_VSLOC
        SDTM: VSLOC when VSTESTCD = SYSBP
        CDASH: SYSBP_VSLOC
    Variable Name: SYSBP_VSORRES
        SDTM: VSORRES when VSTESTCD = SYSBP
        CDASH: SYSBP_VSORRES
    Variable Name: SYSBP_VSORRESU
        SDTM: VSORRESU = mmHg when VSTESTCD = SYSBP
        CDASH: SYSBP_VSORRESU
Concept Name: Diastolic Blood Pressure (Denormalized)
    Variable Name: VSDAT
        SDTM: VSDTC
        CDASH: VSDAT
    Variable Name: DIABP_VSPOS
        SDTM: VSPOS when VSTESTCD = DIABP
        CDASH: DIABP_VSPOS
    Variable Name:

Now we'll transition to validating the ODM file and running a few additional quality checks.

The first step is to import what we need to check validation and conformance, including the odmlib error types.

In [14]:
from odmlib import (
    odm_parser as P,
    OdmlibOIDError,
    OdmlibConformanceError,
    OdmlibElementOrderError,
    create_oid_checker
)

The first step is to create the schema validator. odmlib has built in schemas for many of the main standards, including ODM v1.3.2, ODM v2.0, Define-XML v2.0, and Define-XML v2.1. Once instantiated, we can list the schema validation errors should any exist. The following code instantiates the schema validator and then captures all the errors so we can see the full list, instead of one error at a time.

In [15]:
validator = P.ODMSchemaValidator(standard="odm", version="2.0")

# schema validate the odm.xml and collect all validation errors
errors = list(validator.xsd.iter_errors(odm_file))
# print out each validation error and report how many are due to the PLACEHOLDER content
print(f"Found {len(errors)} schema validation errors:\n")
for i, error in enumerate(errors, 1):
    print(f"{i}. {error.reason}")
    print(f"   Path: {error.path}")
    print()


Found 0 schema validation errors:



Now that basic schema conformance has been verified, we can run some additional, and very useful, check to examine the references in the ODM file and make sure they point to an existing definition. Basically, we check to see if every Ref has an associated Def.

In [16]:
# ensure every ref has an associated def
oid_checker = create_oid_checker("odm_2_0")
try:
    mdv.verify_oids(oid_checker)
except OdmlibOIDError as ve:
    print(f"Error verifying OIDs: {ve}")
else:
    print(f"OIDs verified as valid")


OIDs verified as valid


Using the same oid_checker, we can also scan for orphans, or definitions that are never referenced. In this case we're looking for Defs that do not have associated Refs.

In [17]:
# find any defs that do not have refs
orphans = mdv.unreferenced_oids(oid_checker)
print(f"found {len(orphans)} missing OID Defs")
if orphans:
    print(f"Orphaned OIDs: {orphans}")


found 2 missing OID Defs
Orphaned OIDs: {'ODM.CDASH.STUDY.MDV': 'MetaDataVersionOID', 'IG_FORM.VS1': 'ItemGroupOID'}


This last block verifies the order of the elements in the ODM file. It ensures the elements are ordered according to the specification. There is a reorder_object() feature that will fix the ordering for you. In fact, when creating your own ODM or Define-XMLs this will happen automatically when odmlib writes out the XML file.

In [18]:
# ensure the ODM structure follows the order defined in the specification
try:
    odm.verify_order()
except ValueError as ve:
    print(f"Error verifying element order in Study. {ve}")
else:
    print(f"Study element order is verified")


Study element order is verified
